In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# ==========================================
# 1. LOAD DATA
# ==========================================
print("Memuat dataset yang sudah bersih...")
# Menggunakan dataset hasil pembersihan dari tahap sebelumnya
df_jawa = pd.read_csv('dataset_pulau_jawa_clean.csv')
print(f"Total data Pulau Jawa (Mentah): {len(df_jawa)} baris")

# ==========================================
# 2. DATA PREPROCESSING
# ==========================================
# Menghapus baris yang memiliki nilai kosong pada kolom target
df_jawa = df_jawa.dropna(subset=['totalScore', 'reviewsCount'])

# Filter reviewsCount > 100
df_jawa = df_jawa[df_jawa['reviewsCount'] > 100].copy()
print(f"Total data setelah preprocessing: {len(df_jawa)} baris\n")

# Menampilkan daftar kota yang tersedia
print("Kota/Kabupaten di Pulau Jawa yang tersedia:")
for i, city in enumerate(sorted(df_jawa['city'].unique()), 1):
    print(f"  {i}. {city}")
print()

# ==========================================
# 3. LOG TRANSFORM PADA REVIEWS COUNT
# ==========================================
# reviewsCount memiliki rentang yang sangat lebar.
# Log transform membantu menyamakan skala dan mengurangi pengaruh outlier.
df_jawa['reviewsCount_log'] = np.log1p(df_jawa['reviewsCount'])

# Definisikan fitur (gunakan hasil log, bukan ulasan mentah)
X = df_jawa[['totalScore', 'reviewsCount_log']]

print("\n--- STATISTIK SETELAH LOG TRANSFORM ---")
print(df_jawa[['reviewsCount', 'reviewsCount_log']].describe().round(4))

# Visualisasi distribusi sebelum & sesudah log transform
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_jawa['reviewsCount'], bins=50, color='coral', edgecolor='black')
axes[0].set_title('Distribusi reviewsCount (Mentah)', fontweight='bold')
axes[0].set_xlabel('Jumlah Ulasan')
axes[0].set_ylabel('Frekuensi')
axes[0].grid(True, linestyle='--', alpha=0.5)

axes[1].hist(df_jawa['reviewsCount_log'], bins=50, color='mediumseagreen', edgecolor='black')
axes[1].set_title('Distribusi reviewsCount_log (Log Transform)', fontweight='bold')
axes[1].set_xlabel('Log(1 + Jumlah Ulasan)')
axes[1].set_ylabel('Frekuensi')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.suptitle('Efek Log Transform pada reviewsCount', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('log_transform_reviewsCount.png', dpi=300, bbox_inches='tight')
print("\n✅ Grafik Log Transform tersimpan sebagai 'log_transform_reviewsCount.png'")
plt.show()

# ==========================================
# 4. FEATURE SCALING (STANDARISASI)
# ==========================================
# K-Means sangat dipengaruhi oleh skala fitur.
scaler = StandardScaler()
X_normalized = scaler.fit_transform(X)

print("\n--- FEATURE SCALING SELESAI ---")
print(f"Shape data hasil normalisasi : {X_normalized.shape}")
print(f"Mean tiap fitur              : {X_normalized.mean(axis=0).round(4)}")
print(f"Std tiap fitur               : {X_normalized.std(axis=0).round(4)}")

# Verifikasi 5 baris pertama data hasil normalisasi
df_check = pd.DataFrame(
    X_normalized,
    columns=['totalScore_normalized', 'reviewsCount_normalized']
)
print("\nContoh data setelah normalisasi:")
print(df_check.head())
print("-" * 50)

# ==========================================
# 5. METODE ELBOW - MENCARI K OPTIMAL (K=2 s.d. K=10)
# ==========================================
print("\n" + "=" * 50)
print("     MENCARI K OPTIMAL DENGAN METODE ELBOW")
print("=" * 50)

k_range = range(2, 11)
inertia_list = []
silhouette_list = []

for k in k_range:
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_temp.fit(X_normalized)

    inertia_list.append(kmeans_temp.inertia_)
    sil = silhouette_score(X_normalized, kmeans_temp.labels_)
    silhouette_list.append(sil)

    print(f"K = {k:2d} | Inertia (WCSS) = {kmeans_temp.inertia_:8.3f} | Silhouette = {sil:.4f}")

# Menentukan K optimal secara matematis
def find_elbow(k_values, inertia_values):
    k_values = list(k_values)
    x1, y1 = k_values[0], inertia_values[0]
    x2, y2 = k_values[-1], inertia_values[-1]
    distances = []
    for i in range(len(k_values)):
        x0, y0 = k_values[i], inertia_values[i]
        num = abs((y2 - y1) * x0 - (x2 - x1) * y0 + x2 * y1 - y2 * x1)
        den = np.sqrt((y2 - y1) ** 2 + (x2 - x1) ** 2)
        distances.append(num / den)
    return k_values[np.argmax(distances)]

best_k_elbow = find_elbow(k_range, inertia_list)
best_k_silhouette = k_range[np.argmax(silhouette_list)]

print("\n" + "-" * 50)
print(f"🎯 K optimal (Metode Elbow)      : K = {best_k_elbow}")
print(f"🎯 K optimal (Silhouette Score)  : K = {best_k_silhouette} "
      f"(Score = {max(silhouette_list):.4f})")
print("-" * 50)

# Visualisasi Elbow & Silhouette
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(list(k_range), inertia_list, 'bo-', linewidth=2, markersize=10)
axes[0].axvline(x=best_k_elbow, color='red', linestyle='--',
                label=f'K optimal = {best_k_elbow}')
axes[0].scatter(best_k_elbow, inertia_list[list(k_range).index(best_k_elbow)],
                color='red', s=250, zorder=5, edgecolor='black', linewidth=2)
axes[0].set_title('Metode Elbow (Inertia / WCSS)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Jumlah Cluster (K)', fontsize=12)
axes[0].set_ylabel('Inertia (WCSS)', fontsize=12)
axes[0].set_xticks(list(k_range))
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend(fontsize=11)

for k, val in zip(k_range, inertia_list):
    axes[0].annotate(f'{val:.1f}', xy=(k, val), xytext=(5, 8),
                     textcoords='offset points', fontsize=9)

axes[1].plot(list(k_range), silhouette_list, 'go-', linewidth=2, markersize=10)
axes[1].axvline(x=best_k_silhouette, color='red', linestyle='--',
                label=f'K optimal = {best_k_silhouette}')
axes[1].scatter(best_k_silhouette,
                silhouette_list[list(k_range).index(best_k_silhouette)],
                color='red', s=250, zorder=5, edgecolor='black', linewidth=2)
axes[1].set_title('Silhouette Score per K', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Jumlah Cluster (K)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_xticks(list(k_range))
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].legend(fontsize=11)

for k, val in zip(k_range, silhouette_list):
    axes[1].annotate(f'{val:.3f}', xy=(k, val), xytext=(5, 8),
                     textcoords='offset points', fontsize=9)

plt.suptitle('Analisis K Optimal untuk K-Means Clustering',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('elbow_method.png', dpi=300, bbox_inches='tight')
print("\n✅ Grafik Elbow tersimpan sebagai 'elbow_method.png'")
plt.show()

# Tabel ringkasan
df_elbow = pd.DataFrame({
    'K': list(k_range),
    'Inertia (WCSS)': [round(v, 3) for v in inertia_list],
    'Silhouette Score': [round(v, 4) for v in silhouette_list]
})
print("\n--- TABEL RINGKASAN METODE ELBOW ---")
print(df_elbow.to_string(index=False))

# ==========================================
# 6. FITTING MODEL K-MEANS UNTUK K=3 DAN K=4
# ==========================================
def fit_and_visualize_kmeans(n_clusters, ax, color_palette):
    print("\n" + "=" * 50)
    print(f"        HASIL CLUSTERING DENGAN K={n_clusters}")
    print("=" * 50)

    kmeans_model = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans_model.fit(X_normalized)

    df_jawa[f'Cluster_K{n_clusters}'] = kmeans_model.labels_

    cluster_means = df_jawa.groupby(f'Cluster_K{n_clusters}')['reviewsCount_log'].mean().sort_values()

    if n_clusters == 3:
        cluster_labels = ['Cluster Khusus/Niche', 'Populer Sedang', 'Populer Tinggi']
    elif n_clusters == 4:
        cluster_labels = ['Niche/Minim Ulasan', 'Sedang-Rendah', 'Sedang-Tinggi', 'Sangat Populer']
    else:
        cluster_labels = [f'Cluster {i+1}' for i in range(n_clusters)]

    cluster_mapping = {cluster_means.index[i]: cluster_labels[i] for i in range(n_clusters)}
    df_jawa[f'Cluster_Name_K{n_clusters}'] = df_jawa[f'Cluster_K{n_clusters}'].map(cluster_mapping)

    sil_score = silhouette_score(X_normalized, df_jawa[f'Cluster_K{n_clusters}'])
    inertia = kmeans_model.inertia_

    print(f"Silhouette Score : {sil_score:.4f}")
    print(f"Inertia (WCSS)   : {inertia:.3f}")
    print(f"\nDistribusi Cluster K={n_clusters}:")
    print(df_jawa[f'Cluster_Name_K{n_clusters}'].value_counts().to_string())
    print("-" * 50)

    sns.scatterplot(
        x='totalScore',
        y='reviewsCount_log',
        hue=f'Cluster_Name_K{n_clusters}',
        palette=color_palette,
        data=df_jawa,
        alpha=0.7,
        s=100,
        edgecolor='black',
        linewidth=0.5,
        ax=ax
    )
    ax.set_title(f'K-Means Clustering K={n_clusters}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Total Score (Rating)', fontsize=11)
    ax.set_ylabel('Log Jumlah Ulasan (log1p)', fontsize=11)
    ax.legend(title='Cluster', fontsize=8, title_fontsize=9, loc='upper right')
    ax.grid(True, linestyle='--', alpha=0.5)

    print(f"\nProfil Cluster K={n_clusters}:")
    cluster_profile = df_jawa.groupby(f'Cluster_Name_K{n_clusters}')[['totalScore', 'reviewsCount', 'reviewsCount_log']].mean().round(2)
    cluster_profile['Jumlah_Destinasi'] = df_jawa[f'Cluster_Name_K{n_clusters}'].value_counts()
    print(cluster_profile)

    return kmeans_model, cluster_mapping


fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Palet K=3
palette_k3 = {
    'Cluster Khusus/Niche': '#2ca02c', # Hijau
    'Populer Sedang': '#ff7f0e',       # Oranye
    'Populer Tinggi': '#1f77b4'        # Biru
}

# Palet K=4
palette_k4 = {
    'Niche/Minim Ulasan': '#2ca02c',
    'Sedang-Rendah': '#ff7f0e',
    'Sedang-Tinggi': '#9467bd',
    'Sangat Populer': '#1f77b4'
}

kmeans_k3, mapping_k3 = fit_and_visualize_kmeans(3, axes[0], palette_k3)
kmeans_k4, mapping_k4 = fit_and_visualize_kmeans(4, axes[1], palette_k4)

# Re-run evaluasi K=4 (Opsional sebagai cross-check)
print("\n" + "=" * 50)
print("     RE-RUN CLUSTERING K=4 SETELAH FEATURE SCALING")
print("=" * 50)
kmeans_k4_rerun = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans_k4_rerun.fit(X_normalized)
labels_k4_rerun = kmeans_k4_rerun.labels_
sil_k4_final = silhouette_score(X_normalized, labels_k4_rerun)
print(f"Silhouette Score K=4 : {sil_k4_final:.4f}")
print(f"Inertia K=4          : {kmeans_k4_rerun.inertia_:.3f}")
print(f"Mapping cluster K=4  : {mapping_k4}")
print("-" * 50)

plt.suptitle('Perbandingan Clustering Wisata Pulau Jawa: K=3 vs K=4',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('kmeans_k3_k4_comparison.png', dpi=300, bbox_inches='tight')
print("\n✅ Grafik perbandingan K=3 dan K=4 tersimpan sebagai 'kmeans_k3_k4_comparison.png'")
plt.show()

print("\n--- SAMPEL HASIL CLUSTERING K=3 ---")
print(df_jawa[['title', 'city', 'totalScore', 'reviewsCount', 'reviewsCount_log', 'Cluster_Name_K3']].head(10).to_string())

print("\n--- SAMPEL HASIL CLUSTERING K=4 ---")
print(df_jawa[['title', 'city', 'totalScore', 'reviewsCount', 'reviewsCount_log', 'Cluster_Name_K4']].head(10).to_string())

# ==========================================
# 7. FITTING MODEL FINAL K=3 (OPSIONAL)
# ==========================================
# Jika Anda memutuskan K=3 sebagai model final definitif, bagian ini memastikan label akhir terpasang
print("\n" + "=" * 50)
print("     FITTING MODEL FINAL K=3 (UNTUK EKSPOR)")
print("=" * 50)

kmeans_k3_final = KMeans(n_clusters=3, random_state=42, n_init=10)
df_jawa['Cluster_Final'] = kmeans_k3_final.fit_predict(X_normalized)

# Kita bisa reuse mapping dari sebelumnya
df_jawa['Cluster_Name_Final'] = df_jawa['Cluster_K3'].map(mapping_k3)

# ==========================================
# 8. EKSPOR HASIL CLUSTERING
# ==========================================
# Membersihkan kolom duplikat agar CSV lebih rapi
cols_to_keep = [col for col in df_jawa.columns if col not in ['Cluster_K3', 'Cluster_K4', 'Cluster_Name_K3', 'Cluster_Name_K4']]
df_export = df_jawa[cols_to_keep]

df_export.to_csv('hasil_clustering_pulau_jawa_lengkap.csv', index=False)
print("\n✅ Hasil lengkap tersimpan sebagai 'hasil_clustering_pulau_jawa_lengkap.csv'")